In [ ]:
import base64
import requests
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

In [ ]:
session = get_active_session()

In [ ]:
'''
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

data = [(
    "5f4d6294-960e-40ba-9c59-61561923506f",
    "f3a4d10a-c723-4c6e-9648-2d74f1d90fc7",
    "3ff54734-0bfa-43d0-8b77-d16a864ad722"
)]

columns = ["CLIENT_ID", "CLIENT_SECRET", "REFRESH_TOKEN"]

df = session.create_dataframe(data, schema=columns)

df.write.mode("overwrite").save_as_table("DWDEV.DATASCIENCE.API_CRM_CREDENTIALS")
'''

In [ ]:
sql_query = """
SELECT CLIENT_ID, CLIENT_SECRET, REFRESH_TOKEN
FROM DWDEV.DATASCIENCE.API_CRM_CREDENTIALS
"""

row = session.sql(sql_query).collect()[0]

CLIENT_ID = row["CLIENT_ID"]
CLIENT_SECRET = row["CLIENT_SECRET"]
ACCESS_TOKEN = row["REFRESH_TOKEN"]
print(ACCESS_TOKEN)

In [ ]:
def save_token_snowflake(new_token):
    session.sql(f"""
        UPDATE DWDEV.DATASCIENCE.API_CRM_CREDENTIALS
        SET REFRESH_TOKEN = '{new_token}'
    """).collect()

In [ ]:
def refresh_token():
    # -------- gera grant code --------
    url = "https://api-sandbox.algartelecom.com.br/oauth/grant-code"

    headers = {
        "Content-Type": "application/json"
    }

    data = {
        "client_id": CLIENT_ID,
        "extraInfo": {},
        "redirect_uri": "http://localhost"
    }

    response = requests.post(url, json=data, headers=headers)
    response.raise_for_status()

    code = response.text.split("=")[1][:-2]

    # -------- gera access token --------
    credentials = f"{CLIENT_ID}:{CLIENT_SECRET}"
    credentials_b64 = base64.b64encode(credentials.encode()).decode()

    url = "https://api-sandbox.algartelecom.com.br/oauth/access-token"

    headers = {
        "Authorization": f"Basic {credentials_b64}",
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json"
    }

    data = {
        "grant_type": "authorization_code",
        "code": code
    }

    response = requests.post(url, headers=headers, data=data)
    response.raise_for_status()
    #print(response.json()["access_token"])
    return response.json()["access_token"]

In [ ]:
'''
def main(client_id, url):
    global ACCESS_TOKEN

    payload = {
        "circuitQuantity": 1,
        "maximumReLoyaltyDate": "2024-09-30 00:00:00.000",
        "workedAllCustomerRoot": "SIM",
        "workLogNote": "Criação via api",
        "consultorDocumentNumber": "02893814603",
        "customerSuccessDocumentNumber": "02893814603",
        "primaryDocumentNumber": "13612744000109",
        "circuits": ["A119099847"],
        "createdBy": "CLEUTORM",
        "positionId": "1016",
        "customerId": 3449299
    }


    def call_api():
        headers = {
            "user-name": "rodrigosp@algar.com.br",
            "client_id": client_id,
            "access_token": ACCESS_TOKEN, #"1fb48176-2e63-4071-a6e1-6afd957d4028",
            "Content-Type": "application/json",
            "Cookie": "TS017785f1=0117335212e207dbf4380e81c7f4d2cf097d2203c8ad04d1b77488e849ea4725d654035b46e2cb2f2e4fd46a66d5e8acc31a34c9d7"
        }

        return requests.post(url, headers=headers, json=payload, timeout=30)

    response = call_api()

    if response.status_code == 401:
        ACCESS_TOKEN = refresh_token()

        save_token_snowflake(ACCESS_TOKEN)

        response = call_api()

    #response.raise_for_status()
    return response.text
'''

In [ ]:
import requests

def main(client_id, url, payload):
    global ACCESS_TOKEN

    def call_api():
        headers = {
            "user-name": "rodrigosp@algar.com.br",
            "client_id": client_id,
            "access_token": ACCESS_TOKEN,
            "Content-Type": "application/json",
            "Cookie": "TS017785f1=0117335212e207dbf4380e81c7f4d2cf097d2203c8ad04d1b77488e849ea4725d654035b46e2cb2f2e4fd46a66d5e8acc31a34c9d7"
        }

        return requests.post(url, headers=headers, json=payload, timeout=30)

    response = call_api()

    if response.status_code == 401:
        ACCESS_TOKEN = refresh_token()
        save_token_snowflake(ACCESS_TOKEN)
        response = call_api()

    return response.text

In [ ]:
if __name__ == "__main__":
    payload = {
        "circuitQuantity": 1,
        "maximumReLoyaltyDate": "2024-09-30 00:00:00.000",
        "workedAllCustomerRoot": "SIM",
        "workLogNote": "Criação via api",
        "consultorDocumentNumber": "02893814603",
        "customerSuccessDocumentNumber": "02893814603",
        "primaryDocumentNumber": "13612744000109",
        "circuits": ["A119099847"],
        "createdBy": "CLEUTORM",
        "positionId": "1016",
        "customerId": 3449299
    }
    
    result = main(
        CLIENT_ID,
        "https://api-sandbox.algartelecom.com.br/algar-telecom/corporate/algar-crm/v1/activityCustomerLoyalty",
        payload
    )
    print(result)

# Modelo

In [ ]:
import base64
import requests
import pandas as pd
import numpy as np
import logging
import json
import time
import traceback
from datetime import datetime
from snowflake.snowpark.context import get_active_session

# ================================================================
# ESTRUTURA DE LOGS EM MEMÓRIA → DATAFRAME
# ================================================================

log_records = []  # Lista que acumula todos os eventos

def add_log(level, etapa, mensagem, status_code=None, elapsed_ms=None, detalhe=None):
    """Registra um evento na lista de logs."""
    log_records.append({
        "TIMESTAMP":   datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        "LEVEL":       level,
        "ETAPA":       etapa,
        "MENSAGEM":    mensagem,
        "STATUS_CODE": status_code,
        "ELAPSED_MS":  round(elapsed_ms, 1) if elapsed_ms is not None else None,
        "DETALHE":     detalhe
    })


def get_log_df():
    """Retorna os logs acumulados como DataFrame pandas."""
    df = pd.DataFrame(log_records, columns=[
        "TIMESTAMP", "LEVEL", "ETAPA", "MENSAGEM", "STATUS_CODE", "ELAPSED_MS", "DETALHE"
    ])
    return df


# ================================================================
# SESSÃO SNOWFLAKE
# ================================================================

try:
    session = get_active_session()
except Exception as e:
    add_log("ERROR", "INICIALIZAÇÃO", f"Falha ao criar sessão Snowflake: {e}", detalhe=traceback.format_exc())
    raise


# ================================================================
# CREDENCIAIS
# ================================================================

sql_query = """
SELECT CLIENT_ID, CLIENT_SECRET, REFRESH_TOKEN
FROM DWDEV.DATASCIENCE.API_CRM_CREDENTIALS
"""

try:
    row           = session.sql(sql_query).collect()[0]
    CLIENT_ID     = row["CLIENT_ID"]
    CLIENT_SECRET = row["CLIENT_SECRET"]
    ACCESS_TOKEN  = row["REFRESH_TOKEN"]
except IndexError:
    add_log("ERROR", "CREDENCIAIS", "Nenhuma credencial encontrada na tabela")
    raise
except Exception as e:
    add_log("ERROR", "CREDENCIAIS", f"Erro ao buscar credenciais: {e}", detalhe=traceback.format_exc())
    raise


# ================================================================
# FUNÇÕES AUXILIARES
# ================================================================

def save_token_snowflake(new_token):
    try:
        session.sql(f"""
            UPDATE DWDEV.DATASCIENCE.API_CRM_CREDENTIALS
            SET REFRESH_TOKEN = '{new_token}'
        """).collect()
    except Exception as e:
        add_log("ERROR", "SAVE_TOKEN", f"Falha ao salvar token: {e}", detalhe=traceback.format_exc())
        raise


def refresh_token():

    # -------- grant code --------
    url_grant     = "https://api-sandbox.algartelecom.com.br/oauth/grant-code"
    headers_grant = {"Content-Type": "application/json"}
    data_grant    = {"client_id": CLIENT_ID, "extraInfo": {}, "redirect_uri": "http://localhost"}

    add_log("INFO", "REFRESH_TOKEN", f"[1/2] POST grant-code | {url_grant}")
    t0 = time.time()
    try:
        response = requests.post(url_grant, json=data_grant, headers=headers_grant, timeout=30)
        elapsed  = (time.time() - t0) * 1000
        ok       = response.ok
        add_log(
            "SUCCESS" if ok else "ERROR",
            "REFRESH_TOKEN",
            f"[1/2] Resposta grant-code recebida",
            status_code=response.status_code,
            elapsed_ms=elapsed,
            detalhe=response.text[:300] if not ok else None
        )
        response.raise_for_status()
        code = response.text.split("=")[1][:-2]

    except requests.exceptions.Timeout:
        raise
    except requests.exceptions.HTTPError as e:
        add_log("ERROR", "REFRESH_TOKEN", f"Erro HTTP grant code: {e}", status_code=response.status_code)
        raise
    except Exception as e:
        add_log("ERROR", "REFRESH_TOKEN", f"Erro inesperado grant code: {e}", detalhe=traceback.format_exc())
        raise

    # -------- access token --------
    credentials     = f"{CLIENT_ID}:{CLIENT_SECRET}"
    credentials_b64 = base64.b64encode(credentials.encode()).decode()
    url_token       = "https://api-sandbox.algartelecom.com.br/oauth/access-token"
    headers_token   = {
        "Authorization": f"Basic {credentials_b64}",
        "Content-Type":  "application/x-www-form-urlencoded",
        "Accept":        "application/json"
    }
    data_token = {"grant_type": "authorization_code", "code": code}

    t0 = time.time()
    try:
        response = requests.post(url_token, headers=headers_token, data=data_token, timeout=30)
        elapsed  = (time.time() - t0) * 1000
        ok       = response.ok
        add_log(
            "SUCCESS" if ok else "ERROR",
            "REFRESH_TOKEN",
            "[2/2] Resposta access-token recebida",
            status_code=response.status_code,
            elapsed_ms=elapsed,
            detalhe=response.text[:300] if not ok else None
        )
        response.raise_for_status()
        new_token = response.json()["access_token"]
        return new_token
    except requests.exceptions.Timeout:
        raise
    except requests.exceptions.HTTPError as e:
        add_log("ERROR", "REFRESH_TOKEN", f"Erro HTTP access token: {e}", status_code=response.status_code)
        raise
    except KeyError:
        add_log("ERROR", "REFRESH_TOKEN", "Campo 'access_token' não encontrado na resposta", detalhe=response.text)
        raise
    except Exception as e:
        add_log("ERROR", "REFRESH_TOKEN", f"Erro inesperado access token: {e}", detalhe=traceback.format_exc())
        raise


# ================================================================
# FUNÇÃO PRINCIPAL
# ================================================================

def main(client_id, url, payload):
    global ACCESS_TOKEN

    add_log("INFO", "API_CRM", f"Iniciando chamada | endpoint: {url.split('/')[-1]}")
    add_log("DEBUG", "API_CRM", f"Payload: {json.dumps(payload, ensure_ascii=False)}")

    def call_api():
        headers = {
            "user-name":    "rodrigosp@algar.com.br",
            "client_id":    client_id,
            "access_token": ACCESS_TOKEN,
            "Content-Type": "application/json",
            "Cookie":       "TS017785f1=0117335212e207dbf4380e81c7f4d2cf097d2203c8ad04d1b77488e849ea4725d654035b46e2cb2f2e4fd46a66d5e8acc31a34c9d7"
        }
        t0   = time.time()
        resp = requests.post(url, headers=headers, json=payload, timeout=30)
        elapsed = (time.time() - t0) * 1000
        return resp, elapsed

    # 1ª tentativa
    try:
        response, elapsed = call_api()
        add_log(
            "SUCCESS" if response.ok else "WARNING" if response.status_code == 401 else "ERROR",
            "API_CRM",
            f"Resposta recebida | tentativa 1/2",
            status_code=response.status_code,
            elapsed_ms=elapsed,
            detalhe=response.text[:300] if not response.ok else None
        )
    except requests.exceptions.Timeout:
        raise
    except requests.exceptions.ConnectionError as e:
        add_log("ERROR", "API_CRM", f"Erro de conexão: {e}")
        raise
    except Exception as e:
        add_log("ERROR", "API_CRM", f"Erro inesperado tentativa 1/2: {e}", detalhe=traceback.format_exc())
        raise

    # Renovação automática se 401
    if response.status_code == 401:
        try:
            ACCESS_TOKEN = refresh_token()
            save_token_snowflake(ACCESS_TOKEN)
        except Exception as e:
            add_log("ERROR", "API_CRM", f"Falha na renovação do token: {e}")
            raise

        add_log("INFO", "API_CRM", "Tentativa 2/2 (token renovado)...")
        try:
            response, elapsed = call_api()
            add_log(
                "SUCCESS" if response.ok else "ERROR",
                "API_CRM",
                "Resposta recebida | tentativa 2/2",
                status_code=response.status_code,
                elapsed_ms=elapsed,
                detalhe=response.text[:300] if not response.ok else None
            )
        except Exception as e:
            add_log("ERROR", "API_CRM", f"Falha na retentativa: {e}", detalhe=traceback.format_exc())
            raise

    if response.ok:
        add_log("SUCCESS", "API_CRM", "Chamada finalizada com sucesso")
    else:
        add_log("ERROR", "API_CRM", f"API retornou erro | body: {response.text[:200]}")

    return response.text


# ================================================================
# EXECUÇÃO
# ================================================================

if __name__ == "__main__":
    add_log("INFO", "EXECUÇÃO", f"Início | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    payload = {
        "circuitQuantity": 1,
        "maximumReLoyaltyDate": "2024-09-30 00:00:00.000",
        "workedAllCustomerRoot": "SIM",
        "workLogNote": "Criação via api",
        "consultorDocumentNumber": "02893814603",
        "customerSuccessDocumentNumber": "02893814603",
        "primaryDocumentNumber": "13612744000109",
        "circuits": ["A119099847"],
        "createdBy": "CLEUTORM",
        "positionId": "1016",
        "customerId": 3449299
    }

    add_log("INFO", "EXECUÇÃO", f"customerId: {payload['customerId']} | circuits: {payload['circuits']}")

    try:
        result = main(
            CLIENT_ID,
            "https://api-sandbox.algartelecom.com.br/algar-telecom/corporate/algar-crm/v1/activityCustomerLoyalty",
            payload
        )

    except Exception as e:
        add_log("ERROR", "EXECUÇÃO", f"Processo finalizado com erro: {e}")
    finally:

        df_logs = get_log_df()


        # Exporta também como CSV
        #csv_path = f"api_crm_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        #df_logs.to_csv(csv_path, index=False, encoding="utf-8-sig")
        #add_log("INFO", "EXECUÇÃO", f"Log exportado: {csv_path}")

        session.sql("""
            CREATE TABLE IF NOT EXISTS DWDEV.DATASCIENCE.API_CRM_LOGS (
                TIMESTAMP   VARCHAR,
                LEVEL       VARCHAR(10),
                ETAPA       VARCHAR(50),
                MENSAGEM    VARCHAR(2000),
                STATUS_CODE NUMBER,
                ELAPSED_MS  FLOAT,
                DETALHE     VARCHAR(4000)
            )
        """).collect()
        
        # Salva o DataFrame de logs na tabela
        snow_df = session.create_dataframe(df_logs)
        snow_df.write.mode("append").save_as_table("DWDEV.DATASCIENCE.API_CRM_LOGS")
        
        add_log("SUCCESS", "SNOWFLAKE_LOG", f"{len(df_logs)} registros de log salvos na tabela API_CRM_LOGS")
        
        df_logs

In [ ]:
SELECT * FROM DWDEV.DATASCIENCE.API_CRM_LOGS

In [ ]:
add_log("INFO", "INICIALIZAÇÃO", "Iniciando sessão Snowflake...")